# Objectives

### English

Re-run the current national team roster matching process using the expanded International Player Database V02.

The main objectives of this notebook are:

- Match current national team players against `international_player_database_v02.csv`.
- Measure overall player coverage.
- Analyze coverage by national team.
- Compare the new results against Current Roster Matching V01.
- Quantify the impact of incorporating club competition histories.
- Generate the final matched roster dataset required to build current team vectors.

### Español

Reejecutar el proceso de matching de los planteles actuales utilizando la versión expandida de la International Player Database V02.

Los principales objetivos de esta notebook son:

- Vincular los jugadores actuales de cada selección con `international_player_database_v02.csv`.
- Medir la cobertura general de jugadores.
- Analizar la cobertura por selección nacional.
- Comparar los nuevos resultados con Current Roster Matching V01.
- Cuantificar el impacto de incorporar historiales de competiciones de clubes.
- Generar el dataset final de planteles vinculados necesario para construir los vectores actuales de selección.

# Notebook Configuration

## Imports

In [1]:
from pathlib import Path
import sys
import unicodedata
import re

import numpy as np
import pandas as pd

## Custom Pd Options

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

## Paths

### Base

In [3]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DIR = DATA_DIR / "raw"

PROJECT_ROOT

WindowsPath('..')

### Initial CSVs

In [67]:
CURRENT_ROSTERS_PATH = (
    RAW_DIR / "current_rosters" / "current_rosters_raw_v01.csv"
)

PLAYER_DATABASE_V02_PATH = (
    PROCESSED_DIR / "international_player_database_v02.csv"
)

NAME_CORRECTIONS_PATH = (
    PROCESSED_DIR / "name_corrections_v01.csv"
)

MATCHING_V01_PATH = (
    PROCESSED_DIR / "current_roster_matching_v01.csv"
)

FLEXIBLE_REVIEW_PATH = (
    PROCESSED_DIR
    / "flexible_name_candidates_v02_prevalidated.csv"
)

### Output CSVs

In [5]:
OUTPUT_MATCHING_PATH = (
    PROCESSED_DIR / "current_roster_matching_v02.csv"
)

OUTPUT_UNMATCHED_PATH = (
    PROCESSED_DIR / "unmatched_roster_players_v02.csv"
)

OUTPUT_COVERAGE_PATH = (
    PROCESSED_DIR / "roster_coverage_report_v02.csv"
)

OUTPUT_COMPARISON_PATH = (
    PROCESSED_DIR / "current_roster_matching_comparison_v01_v02.csv"
)

### Checks

In [69]:
required_paths = {
    "Current rosters": CURRENT_ROSTERS_PATH,
    "International Player Database V02": PLAYER_DATABASE_V02_PATH,
    "Name corrections V01": NAME_CORRECTIONS_PATH,
    "Current roster matching V01": MATCHING_V01_PATH,
}

for file_name, file_path in required_paths.items():
    print(f"{file_name}:")
    print(f"  Path: {file_path}")
    print(f"  Exists: {file_path.exists()}")

Current rosters:
  Path: ..\data\raw\current_rosters\current_rosters_raw_v01.csv
  Exists: True
International Player Database V02:
  Path: ..\data\processed\international_player_database_v02.csv
  Exists: True
Name corrections V01:
  Path: ..\data\processed\name_corrections_v01.csv
  Exists: True
Current roster matching V01:
  Path: ..\data\processed\current_roster_matching_v01.csv
  Exists: True


In [68]:
required_paths["Flexible candidate review"] = FLEXIBLE_REVIEW_PATH

In [7]:
missing_paths = [
    file_path
    for file_path in required_paths.values()
    if not file_path.exists()
]

assert not missing_paths, (
    "The following required files were not found:\n"
    + "\n".join(str(path) for path in missing_paths)
)

print("All required input files were found.")

All required input files were found.


## Load Datasets

In [8]:
current_rosters = pd.read_csv(CURRENT_ROSTERS_PATH)

player_database_v02 = pd.read_csv(
    PLAYER_DATABASE_V02_PATH
)

name_corrections = pd.read_csv(
    NAME_CORRECTIONS_PATH
)

matching_v01 = pd.read_csv(
    MATCHING_V01_PATH
)

In [9]:
print("Current rosters:", current_rosters.shape)
print("International Player Database V02:", player_database_v02.shape)
print("Name corrections:", name_corrections.shape)
print("Current roster matching V01:", matching_v01.shape)

Current rosters: (1248, 4)
International Player Database V02: (4058, 33)
Name corrections: (75, 7)
Current roster matching V01: (1248, 10)


### Display

In [10]:
print("current_rosters")
display(current_rosters.head())
print("\n")

print("player_database_v02")
display(player_database_v02.head())
print("\n")

print("name_corrections")
display(name_corrections.head())
print("\n")

print("matching_v01")
display(matching_v01.head())
print("\n")


current_rosters


,country,player_name,position,shirt_number
0,Algeria (ALG),MASTIL Melvin,Goalkeeper,1
1,Algeria (ALG),MANDI Aissa,Defender,2
2,Algeria (ALG),ABADA Achref,Defender,3
3,Algeria (ALG),TOUGAI Mohamed,Defender,4
4,Algeria (ALG),BELAID Zineddine,Defender,5




player_database_v02


,player_id,player_name,team_name,matches_played,competitions_played,seasons_played,50/50,Bad Behaviour,Ball Receipt*,Ball Recovery,Block,Carry,Clearance,Dispossessed,Dribble,Dribbled Past,Duel,Foul Committed,Foul Won,Goal Keeper,Interception,Miscontrol,Pass,Player Off,Player On,Pressure,Shield,Shot,Error,Offside,Own Goal Against,Camera On,Own Goal For
0,2935,Nordi Mukiele Mulere,Paris Saint-Germain,0.0,0.0,0.0,3.0,0.0,399,26,16,344,14,10,8,7,25,14,7,0,13,4,455,2.0,2.0,107,4.0,5,0.0,0.0,0.0,0.0,0.0
1,2936,Christophe Kerbrat,Guingamp,0.0,0.0,0.0,0.0,0.0,12,3,2,13,7,0,0,3,6,2,1,0,2,0,28,0.0,0.0,18,0.0,0,0.0,0.0,0.0,0.0,0.0
2,2937,Christ-Emmanuel Faitout Maouassa,Montpellier,0.0,0.0,0.0,0.0,0.0,50,5,1,42,0,1,5,4,3,3,4,0,3,2,44,0.0,0.0,27,0.0,0,0.0,0.0,0.0,0.0,0.0
3,2940,Jean Eudès Aholou,Strasbourg,0.0,0.0,0.0,0.0,0.0,9,1,0,9,0,0,0,0,0,2,1,0,2,0,9,0.0,0.0,4,0.0,0,0.0,0.0,0.0,0.0,0.0
4,2941,Ismaïla Sarr,Senegal,11.0,2.0,3.0,5.0,0.0,547,39,8,384,8,27,42,9,30,14,56,0,5,36,296,2.0,2.0,250,0.0,30,0.0,1.0,0.0,0.0,0.0




name_corrections


,country,roster_name,roster_name_normalized,database_name,database_name_normalized,correction_type,notes
0,Algeria (ALG),BOUDAOUI Hicham,boudaoui hicham,Hichem Boudaoui,hichem boudaoui,spelling/accent,manually proposed from roster/database country...
1,Argentina (ARG),GONZALEZ Nico,gonzalez nico,Nicolás Iván González,nicolas ivan gonzalez,nickname/full_name,manually proposed from roster/database country...
2,Austria (AUT),MWENE Phillip,mwene phillip,Philipp Mwene,philipp mwene,spelling,manually proposed from roster/database country...
3,Austria (AUT),SCHOEPF Alessandro,schoepf alessandro,Alessandro Schöpf,alessandro schopf,accent/transliteration,manually proposed from roster/database country...
4,Brazil (BRA),MARQUINHOS Marcos,marquinhos marcos,Marcos Aoás Corrêa,marcos aoas correa,nickname/full_name,manually proposed from roster/database country...




matching_v01


,country,country_clean,roster_name,position,shirt_number,player_id,database_name,database_team,matched,match_method
0,Algeria (ALG),Algeria,MASTIL Melvin,Goalkeeper,1,NaN,NaN,NaN,False,unmatched
1,Algeria (ALG),Algeria,MANDI Aissa,Defender,2,6648.0,Aïssa Mandi,Algeria,True,token_subset_country
2,Algeria (ALG),Algeria,ABADA Achref,Defender,3,NaN,NaN,NaN,False,unmatched
3,Algeria (ALG),Algeria,TOUGAI Mohamed,Defender,4,160943.0,Mohamed Amine Tougai,Algeria,True,token_subset_country
4,Algeria (ALG),Algeria,BELAID Zineddine,Defender,5,NaN,NaN,NaN,False,unmatched


### Columns

In [11]:
datasets = {
    "Current rosters": current_rosters,
    "Player database V02": player_database_v02,
    "Name corrections": name_corrections,
    "Matching V01": matching_v01,
}

for dataset_name, dataframe in datasets.items():
    print(f"\n{dataset_name} columns:")
    print(dataframe.columns.tolist())


Current rosters columns:
['country', 'player_name', 'position', 'shirt_number']

Player database V02 columns:
['player_id', 'player_name', 'team_name', 'matches_played', 'competitions_played', 'seasons_played', '50/50', 'Bad Behaviour', 'Ball Receipt*', 'Ball Recovery', 'Block', 'Carry', 'Clearance', 'Dispossessed', 'Dribble', 'Dribbled Past', 'Duel', 'Foul Committed', 'Foul Won', 'Goal Keeper', 'Interception', 'Miscontrol', 'Pass', 'Player Off', 'Player On', 'Pressure', 'Shield', 'Shot', 'Error', 'Offside', 'Own Goal Against', 'Camera On', 'Own Goal For']

Name corrections columns:
['country', 'roster_name', 'roster_name_normalized', 'database_name', 'database_name_normalized', 'correction_type', 'notes']

Matching V01 columns:
['country', 'country_clean', 'roster_name', 'position', 'shirt_number', 'player_id', 'database_name', 'database_team', 'matched', 'match_method']


### Checks

In [12]:
assert not current_rosters.empty
assert not player_database_v02.empty
assert not matching_v01.empty

assert "player_id" in player_database_v02.columns
assert "player_name" in player_database_v02.columns
assert "team_name" in player_database_v02.columns

assert player_database_v02["player_id"].is_unique, (
    "player_id must be unique in International Player Database V02."
)

print("Initial structural checks passed.")

Initial structural checks passed.


## Utility Functions

In [13]:
def normalize_name(name):
    """
    Normalize player names for matching.

    - Converts to lowercase.
    - Removes accents.
    - Removes punctuation.
    - Normalizes whitespace.
    """
    if pd.isna(name):
        return ""

    name = str(name).lower().strip()

    name = unicodedata.normalize("NFKD", name)
    name = "".join(char for char in name if not unicodedata.combining(char))

    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name


In [14]:
def clean_roster_country(country):
    return re.sub(r"\s*\([A-Z]{3}\)$", "", country).strip()

In [15]:
def build_name_lookup(player_database):
    """
    Build a normalized player-name lookup and record how many database
    players share each normalized name.
    """
    lookup = player_database.copy()

    lookup["database_name_normalized"] = (
        lookup["player_name"]
        .fillna("")
        .map(normalize_name)
    )

    lookup["normalized_name_frequency"] = (
        lookup.groupby("database_name_normalized")["player_id"]
        .transform("nunique")
    )

    return lookup

In [16]:
def create_name_signature(name):
    """
    Create an order-independent name representation by sorting
    the normalized name tokens alphabetically.
    """
    normalized_name = normalize_name(name)
    tokens = normalized_name.split()

    return " ".join(sorted(tokens))

In [17]:
def tokens_are_compatible(
    roster_token,
    database_token,
    min_prefix_length=3,
):
    """
    Return True when two normalized tokens are equal or when one token
    is a sufficiently long prefix of the other.
    """
    if roster_token == database_token:
        return True

    if min(len(roster_token), len(database_token)) < min_prefix_length:
        return False

    return (
        database_token.startswith(roster_token)
        or roster_token.startswith(database_token)
    )

In [18]:
def is_flexible_name_candidate(
    roster_tokens,
    database_name,
    min_prefix_length=3,
):
    """
    Check whether every roster-name token is compatible with at least
    one token from a normalized database name.
    """
    database_tokens = database_name.split()

    return all(
        any(
            tokens_are_compatible(
                roster_token,
                database_token,
                min_prefix_length=min_prefix_length,
            )
            for database_token in database_tokens
        )
        for roster_token in roster_tokens
    )

In [19]:
def find_flexible_token_candidates(
    roster_name,
    player_database,
    min_prefix_length=3,
):
    """
    Return all plausible database candidates for a roster player.

    This function only generates candidates. It does not automatically
    assign a player identity.
    """
    roster_name_normalized = normalize_name(roster_name)
    roster_tokens = roster_name_normalized.split()

    if not roster_tokens:
        return player_database.iloc[0:0].copy()

    candidate_mask = (
        player_database["database_name_normalized"]
        .apply(
            lambda database_name: is_flexible_name_candidate(
                roster_tokens=roster_tokens,
                database_name=database_name,
                min_prefix_length=min_prefix_length,
            )
        )
    )

    return player_database.loc[candidate_mask].copy()

# Phase 1: Automatic Matching Experiment

## Configuration

### Prepare Current Rosters

In [20]:
prepared_rosters = current_rosters.copy()

prepared_rosters["country_clean"] = (
    prepared_rosters["country"]
    .map(clean_roster_country)
)

prepared_rosters["roster_name"] = (
    prepared_rosters["player_name"]
)

prepared_rosters["roster_name_normalized"] = (
    prepared_rosters["roster_name"]
    .fillna("")
    .map(normalize_name)
)

prepared_rosters = prepared_rosters[
    [
        "country",
        "country_clean",
        "roster_name",
        "roster_name_normalized",
        "position",
        "shirt_number",
    ]
].copy()

prepared_rosters.head()

,country,country_clean,roster_name,roster_name_normalized,position,shirt_number
0,Algeria (ALG),Algeria,MASTIL Melvin,mastil melvin,Goalkeeper,1
1,Algeria (ALG),Algeria,MANDI Aissa,mandi aissa,Defender,2
2,Algeria (ALG),Algeria,ABADA Achref,abada achref,Defender,3
3,Algeria (ALG),Algeria,TOUGAI Mohamed,tougai mohamed,Defender,4
4,Algeria (ALG),Algeria,BELAID Zineddine,belaid zineddine,Defender,5


### Checks

In [21]:
assert len(prepared_rosters) == len(current_rosters)

assert prepared_rosters["roster_name_normalized"].ne("").all(), (
    "Some roster names became empty after normalization."
)

assert prepared_rosters["country_clean"].notna().all(), (
    "Some roster countries could not be standardized."
)

print("Current rosters prepared:", prepared_rosters.shape)

Current rosters prepared: (1248, 6)


### Prepare Player Database V02

In [22]:
player_lookup_v02 = build_name_lookup(
    player_database_v02
)

player_lookup_v02[
    [
        "player_id",
        "player_name",
        "database_name_normalized",
        "team_name",
        "normalized_name_frequency",
    ]
].head()

,player_id,player_name,database_name_normalized,team_name,normalized_name_frequency
0,2935,Nordi Mukiele Mulere,nordi mukiele mulere,Paris Saint-Germain,1
1,2936,Christophe Kerbrat,christophe kerbrat,Guingamp,1
2,2937,Christ-Emmanuel Faitout Maouassa,christ emmanuel faitout maouassa,Montpellier,1
3,2940,Jean Eudès Aholou,jean eudes aholou,Strasbourg,1
4,2941,Ismaïla Sarr,ismaila sarr,Senegal,1


In [23]:
assert player_lookup_v02["player_id"].is_unique

assert player_lookup_v02["database_name_normalized"].ne("").all(), (
    "Some database names became empty after normalization."
)

print("Player Database V02 lookup:", player_lookup_v02.shape)

Player Database V02 lookup: (4058, 35)


## Analyze unique normalized names 

In [24]:
name_uniqueness_summary = pd.DataFrame(
    {
        "metric": [
            "Database players",
            "Unique normalized names",
            "Players with unique normalized names",
            "Players sharing a normalized name",
        ],
        "value": [
            len(player_lookup_v02),
            player_lookup_v02["database_name_normalized"].nunique(),
            player_lookup_v02[
                "normalized_name_frequency"
            ].eq(1).sum(),
            player_lookup_v02[
                "normalized_name_frequency"
            ].gt(1).sum(),
        ],
    }
)

name_uniqueness_summary

,metric,value
0,Database players,4058
1,Unique normalized names,4058
2,Players with unique normalized names,4058
3,Players sharing a normalized name,0


In [25]:
duplicate_normalized_names = (
    player_lookup_v02
    .loc[
        player_lookup_v02["normalized_name_frequency"].gt(1),
        [
            "database_name_normalized",
            "player_id",
            "player_name",
            "team_name",
        ],
    ]
    .sort_values(
        ["database_name_normalized", "player_name"]
    )
)

print(
    "Players involved in normalized-name collisions:",
    len(duplicate_normalized_names)
)

display(duplicate_normalized_names.head(20))


Players involved in normalized-name collisions: 0


,database_name_normalized,player_id,player_name,team_name


## Define Player Lookup

In [26]:
player_lookup = (
    player_lookup_v02
    .set_index("database_name_normalized")
)

In [27]:
player_lookup.head()

,player_id,player_name,team_name,matches_played,competitions_played,seasons_played,50/50,Bad Behaviour,Ball Receipt*,Ball Recovery,Block,Carry,Clearance,Dispossessed,Dribble,Dribbled Past,Duel,Foul Committed,Foul Won,Goal Keeper,Interception,Miscontrol,Pass,Player Off,Player On,Pressure,Shield,Shot,Error,Offside,Own Goal Against,Camera On,Own Goal For,normalized_name_frequency
database_name_normalized,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
nordi mukiele mulere,2935,Nordi Mukiele Mulere,Paris Saint-Germain,0.0,0.0,0.0,3.0,0.0,399,26,16,344,14,10,8,7,25,14,7,0,13,4,455,2.0,2.0,107,4.0,5,0.0,0.0,0.0,0.0,0.0,1
christophe kerbrat,2936,Christophe Kerbrat,Guingamp,0.0,0.0,0.0,0.0,0.0,12,3,2,13,7,0,0,3,6,2,1,0,2,0,28,0.0,0.0,18,0.0,0,0.0,0.0,0.0,0.0,0.0,1
christ emmanuel faitout maouassa,2937,Christ-Emmanuel Faitout Maouassa,Montpellier,0.0,0.0,0.0,0.0,0.0,50,5,1,42,0,1,5,4,3,3,4,0,3,2,44,0.0,0.0,27,0.0,0,0.0,0.0,0.0,0.0,0.0,1
jean eudes aholou,2940,Jean Eudès Aholou,Strasbourg,0.0,0.0,0.0,0.0,0.0,9,1,0,9,0,0,0,0,0,2,1,0,2,0,9,0.0,0.0,4,0.0,0,0.0,0.0,0.0,0.0,0.0,1
ismaila sarr,2941,Ismaïla Sarr,Senegal,11.0,2.0,3.0,5.0,0.0,547,39,8,384,8,27,42,9,30,14,56,0,5,36,296,2.0,2.0,250,0.0,30,0.0,1.0,0.0,0.0,0.0,1


### Select Necessary Columns

In [28]:
lookup_columns = [
    "player_id",
    "player_name",
    "team_name",
]

exact_match_lookup = player_lookup[lookup_columns].rename(
    columns={
        "player_name": "database_name",
        "team_name": "database_team",
    }
)

exact_match_lookup.head()

,player_id,database_name,database_team
database_name_normalized,,,
nordi mukiele mulere,2935,Nordi Mukiele Mulere,Paris Saint-Germain
christophe kerbrat,2936,Christophe Kerbrat,Guingamp
christ emmanuel faitout maouassa,2937,Christ-Emmanuel Faitout Maouassa,Montpellier
jean eudes aholou,2940,Jean Eudès Aholou,Strasbourg
ismaila sarr,2941,Ismaïla Sarr,Senegal


### Join Rosters with Lookup

In [29]:
matched_rosters_v02 = prepared_rosters.join(
    exact_match_lookup,
    on="roster_name_normalized",
    how="left",
)

In [30]:
matched_rosters_v02["matched"] = (
    matched_rosters_v02["player_id"].notna()
)

matched_rosters_v02["match_method"] = np.where(
    matched_rosters_v02["matched"],
    "exact_normalized_name",
    "unmatched",
)

In [31]:
matched_rosters_v02.head()

,country,country_clean,roster_name,roster_name_normalized,position,shirt_number,player_id,database_name,database_team,matched,match_method
0,Algeria (ALG),Algeria,MASTIL Melvin,mastil melvin,Goalkeeper,1,NaN,NaN,NaN,False,unmatched
1,Algeria (ALG),Algeria,MANDI Aissa,mandi aissa,Defender,2,NaN,NaN,NaN,False,unmatched
2,Algeria (ALG),Algeria,ABADA Achref,abada achref,Defender,3,NaN,NaN,NaN,False,unmatched
3,Algeria (ALG),Algeria,TOUGAI Mohamed,tougai mohamed,Defender,4,NaN,NaN,NaN,False,unmatched
4,Algeria (ALG),Algeria,BELAID Zineddine,belaid zineddine,Defender,5,NaN,NaN,NaN,False,unmatched


In [32]:
matched_rosters_v02 = matched_rosters_v02[
    [
        "country",
        "country_clean",
        "roster_name",
        "roster_name_normalized",
        "position",
        "shirt_number",
        "player_id",
        "database_name",
        "database_team",
        "matched",
        "match_method",
    ]
].copy()

In [33]:
assert len(matched_rosters_v02) == len(prepared_rosters), (
    "The matching process changed the number of roster players."
)

assert matched_rosters_v02["matched"].equals(
    matched_rosters_v02["player_id"].notna()
)

assert matched_rosters_v02.loc[
    matched_rosters_v02["matched"],
    "database_name",
].notna().all()

print("Automatic exact matching completed.")
print("Roster players:", len(matched_rosters_v02))
print("Matched:", matched_rosters_v02["matched"].sum())
print("Unmatched:", (~matched_rosters_v02["matched"]).sum())

Automatic exact matching completed.
Roster players: 1248
Matched: 11
Unmatched: 1237


I procceed to create "signature name"

## Apply Name Signature Method

In [34]:
print(create_name_signature("MANDI Aissa"))
print(create_name_signature("Aïssa Mandi"))

aissa mandi
aissa mandi


In [35]:
prepared_rosters["roster_name_signature"] = (
    prepared_rosters["roster_name"]
    .map(create_name_signature)
)

player_lookup_v02["database_name_signature"] = (
    player_lookup_v02["player_name"]
    .map(create_name_signature)
)

In [36]:
signature_frequency = (
    player_lookup_v02
    .groupby("database_name_signature")["player_id"]
    .transform("nunique")
)

player_lookup_v02["name_signature_frequency"] = signature_frequency

In [37]:
signature_summary = pd.DataFrame(
    {
        "metric": [
            "Unique name signatures",
            "Players with unique signatures",
            "Players sharing a signature",
        ],
        "value": [
            player_lookup_v02["database_name_signature"].nunique(),
            (
                player_lookup_v02["name_signature_frequency"] == 1
            ).sum(),
            (
                player_lookup_v02["name_signature_frequency"] > 1
            ).sum(),
        ],
    }
)

signature_summary

,metric,value
0,Unique name signatures,4058
1,Players with unique signatures,4058
2,Players sharing a signature,0


In [38]:
duplicate_signatures = (
    player_lookup_v02
    .loc[
        player_lookup_v02["name_signature_frequency"] > 1,
        [
            "database_name_signature",
            "player_id",
            "player_name",
            "team_name",
        ],
    ]
    .sort_values(
        ["database_name_signature", "player_name"]
    )
)

print("Players involved in signature collisions:", len(duplicate_signatures))

display(duplicate_signatures.head(20))

Players involved in signature collisions: 0


,database_name_signature,player_id,player_name,team_name


## Build Signature Name Lookup

In [39]:
signature_lookup = (
    player_lookup_v02
    .loc[
        player_lookup_v02["name_signature_frequency"] == 1,
        [
            "database_name_signature",
            "player_id",
            "player_name",
            "team_name",
        ],
    ]
    .set_index("database_name_signature")
    .rename(
        columns={
            "player_name": "database_name",
            "team_name": "database_team",
        }
    )
)

In [40]:
signature_matches = prepared_rosters[
    [
        "roster_name_signature",
    ]
].join(
    signature_lookup,
    on="roster_name_signature",
    how="left",
    rsuffix="_signature",
)

### Unmatched Mask

In [41]:
unmatched_mask = ~matched_rosters_v02["matched"]

signature_match_mask = (
    unmatched_mask
    & signature_matches["player_id"].notna()
)

matched_rosters_v02.loc[
    signature_match_mask,
    "player_id",
] = signature_matches.loc[
    signature_match_mask,
    "player_id",
]

matched_rosters_v02.loc[
    signature_match_mask,
    "database_name",
] = signature_matches.loc[
    signature_match_mask,
    "database_name",
]

matched_rosters_v02.loc[
    signature_match_mask,
    "database_team",
] = signature_matches.loc[
    signature_match_mask,
    "database_team",
]

matched_rosters_v02.loc[
    signature_match_mask,
    "matched",
] = True

matched_rosters_v02.loc[
    signature_match_mask,
    "match_method",
] = "exact_name_tokens"

## Evaluate Experimental Results

In [42]:
print("Matching after token-order normalization:")
print("Roster players:", len(matched_rosters_v02))
print("Matched:", matched_rosters_v02["matched"].sum())
print("Unmatched:", (~matched_rosters_v02["matched"]).sum())
print(f"Coverage: {matched_rosters_v02['matched'].mean():.2%}")

Matching after token-order normalization:
Roster players: 1248
Matched: 309
Unmatched: 939
Coverage: 24.76%


In [43]:
matched_rosters_v02["match_method"].value_counts()

match_method
unmatched                939
exact_name_tokens        298
exact_normalized_name     11
Name: count, dtype: int64

# Phase 2: Initialize Matching V02 from Validated V01 Results

Current Roster Matching V01 already contains player identities recovered through exact matching, token-based matching, and validated manual corrections.

To isolate the effect of expanding the historical player database, these previously validated matches are preserved. New matching attempts are restricted to players that remained unmatched in V01.

In [44]:
matched_rosters_v02 = matching_v01.copy()

print("Roster players:", len(matched_rosters_v02))
print("Previously matched:", matched_rosters_v02["matched"].sum())
print("Previously unmatched:", (~matched_rosters_v02["matched"]).sum())
print(f"V01 coverage: {matched_rosters_v02['matched'].mean():.2%}")

Roster players: 1248
Previously matched: 577
Previously unmatched: 671
V01 coverage: 46.23%


In [45]:
matched_v01_ids = (
    matched_rosters_v02
    .loc[matched_rosters_v02["matched"], "player_id"]
    .dropna()
    .astype(int)
)

missing_v01_ids_in_v02 = (
    set(matched_v01_ids)
    - set(player_database_v02["player_id"].astype(int))
)

print(
    "Validated V01 player IDs missing from Database V02:",
    len(missing_v01_ids_in_v02),
)

assert not missing_v01_ids_in_v02, (
    "Some validated V01 player IDs are missing from Player Database V02."
)

Validated V01 player IDs missing from Database V02: 0


In [46]:
previously_unmatched = (
    matched_rosters_v02
    .loc[~matched_rosters_v02["matched"]]
    .copy()
)

previously_unmatched["roster_name_normalized"] = (
    previously_unmatched["roster_name"]
    .map(normalize_name)
)

previously_unmatched["roster_name_signature"] = (
    previously_unmatched["roster_name"]
    .map(create_name_signature)
)

print("Previously unmatched players:", len(previously_unmatched))
previously_unmatched.head()

Previously unmatched players: 671


,country,country_clean,roster_name,position,shirt_number,player_id,database_name,database_team,matched,match_method,roster_name_normalized,roster_name_signature
0,Algeria (ALG),Algeria,MASTIL Melvin,Goalkeeper,1,NaN,NaN,NaN,False,unmatched,mastil melvin,mastil melvin
2,Algeria (ALG),Algeria,ABADA Achref,Defender,3,NaN,NaN,NaN,False,unmatched,abada achref,abada achref
4,Algeria (ALG),Algeria,BELAID Zineddine,Defender,5,NaN,NaN,NaN,False,unmatched,belaid zineddine,belaid zineddine
8,Algeria (ALG),Algeria,GOUIRI Amine,Forward,9,NaN,NaN,NaN,False,unmatched,gouiri amine,amine gouiri
10,Algeria (ALG),Algeria,HADJ MOUSSA,Forward,11,NaN,NaN,NaN,False,unmatched,hadj moussa,hadj moussa


## Recover Previously Unmatched Players

### Normalized Name Lookup

In [47]:
normalized_name_results = (
    player_lookup[
        [
            "player_id",
            "player_name",
            "team_name",
        ]
    ]
    .reindex(
        previously_unmatched["roster_name_normalized"]
    )
)

In [48]:
normalized_name_results.index = previously_unmatched.index

In [49]:
normalized_name_results = normalized_name_results.rename(
    columns={
        "player_name": "database_name",
        "team_name": "database_team",
    }
)

In [50]:
normalized_name_results.head()

,player_id,database_name,database_team
0,NaN,NaN,NaN
2,NaN,NaN,NaN
4,NaN,NaN,NaN
8,NaN,NaN,NaN
10,NaN,NaN,NaN


In [51]:
normalized_name_results["player_id"].notna().sum()

np.int64(0)

In [52]:
roster_normalized_names = set(
    previously_unmatched["roster_name_normalized"]
)

database_normalized_names = set(
    player_lookup.index
)

exact_name_intersection = (
    roster_normalized_names
    & database_normalized_names
)

print(
    "Exact normalized-name intersections:",
    len(exact_name_intersection),
)

list(exact_name_intersection)[:20]

Exact normalized-name intersections: 0


[]

In [53]:
exact_normalized_matches = (
    normalized_name_results["player_id"]
    .notna()
    .sum()
)

print(
    "New exact normalized-name matches:",
    exact_normalized_matches,
)

New exact normalized-name matches: 0


In [54]:
remaining_unmatched = previously_unmatched.copy()

In [55]:
signature_lookup = (
    player_lookup_v02
    .loc[
        player_lookup_v02[
            "name_signature_frequency"
        ] == 1,
        [
            "database_name_signature",
            "player_id",
            "player_name",
            "team_name",
        ],
    ]
    .set_index("database_name_signature")
    .rename(
        columns={
            "player_name": "database_name",
            "team_name": "database_team",
        }
    )
)

In [56]:
signature_results = (
    signature_lookup[
        [
            "player_id",
            "database_name",
            "database_team",
        ]
    ]
    .reindex(
        remaining_unmatched["roster_name_signature"]
    )
)

signature_results.index = remaining_unmatched.index

In [57]:
signature_match_mask = (
    signature_results["player_id"].notna()
)

print(
    "New token-signature matches:",
    signature_match_mask.sum(),
)

print(
    "Still unmatched:",
    (~signature_match_mask).sum(),
)

New token-signature matches: 38
Still unmatched: 633


## Export Still Unmatched

I will proceed to analyze them manually instead of making an algorithm.

In [58]:
remaining_unmatched = (
    matched_rosters_v02
    .loc[~matched_rosters_v02["matched"]]
    .copy()
)

remaining_unmatched.to_csv(
    OUTPUT_UNMATCHED_PATH,
    index=False
)

print(f"Remaining unmatched players: {len(remaining_unmatched)}")
print(f"Exported to: {OUTPUT_UNMATCHED_PATH}")

Remaining unmatched players: 671
Exported to: ..\data\processed\unmatched_roster_players_v02.csv


### EDA

In [59]:
(
    remaining_unmatched["country"]
    .value_counts()
    .sort_values(ascending=False)
)

country
Bosnia And Herzegovina (BIH)    26
Curaçao (CUW)                   26
Haiti (HAI)                     26
Iraq (IRQ)                      26
Jordan (JOR)                    26
New Zealand (NZL)               26
Norway (NOR)                    26
Uzbekistan (UZB)                26
Sweden (SWE)                    24
Australia (AUS)                 18
Morocco (MAR)                   18
Tunisia (TUN)                   18
Saudi Arabia (KSA)              17
Paraguay (PAR)                  16
Algeria (ALG)                   15
Korea Republic (KOR)            15
South Africa (RSA)              15
Egypt (EGY)                     14
Ghana (GHA)                     14
IR Iran (IRN)                   14
Japan (JPN)                     14
Qatar (QAT)                     14
Côte D'Ivoire (CIV)             13
Cabo Verde (CPV)                12
Belgium (BEL)                   11
Croatia (CRO)                   11
France (FRA)                    11
Germany (GER)                   11
Netherlands 

In [60]:
remaining_unmatched["token_count"] = (
    remaining_unmatched["roster_name"]
    .str.split()
    .str.len()
)

remaining_unmatched["token_count"].value_counts().sort_index()

token_count
2    671
Name: count, dtype: int64

In [61]:
remaining_unmatched["roster_name"].str.len().describe()

count    671.000000
mean      13.298063
std        2.738991
min        6.000000
25%       11.000000
50%       13.000000
75%       15.000000
max       24.000000
Name: roster_name, dtype: float64

In [62]:
display(
    remaining_unmatched[
        [
            "country",
            "roster_name",
        ]
    ]
    .head(100)
)

,country,roster_name
0,Algeria (ALG),MASTIL Melvin
2,Algeria (ALG),ABADA Achref
4,Algeria (ALG),BELAID Zineddine
8,Algeria (ALG),GOUIRI Amine
10,Algeria (ALG),HADJ MOUSSA
11,Algeria (ALG),BENBOUALI Nadhir
12,Algeria (ALG),HADJAM Jaouen
15,Algeria (ALG),BENBOT Oussama
16,Algeria (ALG),BELGHALI Ra
19,Algeria (ALG),BOULBINA Adil


#### Checkpoint - English

##### Matching Strategy After Exploratory Analysis

The exploratory analysis shows that the remaining unmatched players cannot be treated as a single homogeneous group.

Several unresolved cases appear to involve:

- Partial first names.
- Nicknames or shortened names.
- Different name-order conventions.
- Minor spelling or transliteration differences.
- Truncated roster names.
- Players that may still be absent from the historical database.

Applying increasingly permissive automatic matching rules could introduce incorrect player identities.

Therefore, the next stage uses flexible token comparison only to generate plausible candidates. No player is matched automatically during this stage. Candidate pairs are exported for explicit review and validation before being incorporated into the final roster matching dataset.

#### Checkpoint - Español

##### Estrategia de Matching después del Análisis Exploratorio

El análisis exploratorio muestra que los jugadores restantes sin identificar no pueden tratarse como un único grupo homogéneo.

Varios de los casos no resueltos parecen involucrar:

- Nombres propios parciales.
- Apodos o nombres abreviados.
- Diferentes convenciones en el orden de los nombres.
- Pequeñas diferencias ortográficas o de transliteración.
- Nombres truncados en el roster.
- Jugadores que todavía podrían estar ausentes de la base histórica.

Aplicar reglas automáticas cada vez más permisivas podría introducir identidades incorrectas.

Por lo tanto, la siguiente etapa utiliza una comparación flexible de tokens únicamente para generar candidatos plausibles. Ningún jugador se identifica automáticamente durante esta etapa. Los pares candidatos se exportan para revisión y validación explícita antes de incorporarlos al dataset final de matching.

## Flexible Candidate Generation

### Generate Candidate Pairs

In [63]:
candidate_rows = []

for roster_index, roster_row in remaining_unmatched.iterrows():

    candidates = find_flexible_token_candidates(
        roster_name=roster_row["roster_name"],
        player_database=player_lookup_v02,
    )

    for _, candidate in candidates.iterrows():

        candidate_rows.append(
            {
                "roster_index": roster_index,
                "country": roster_row["country"],
                "roster_name": roster_row["roster_name"],
                "candidate_player_id": candidate["player_id"],
                "candidate_name": candidate["player_name"],
                "candidate_team": candidate["team_name"],
            }
        )

flexible_candidates = pd.DataFrame(candidate_rows)

print(
    f"Candidate pairs generated: {len(flexible_candidates)}"
)

flexible_candidates.head()

Candidate pairs generated: 93


,roster_index,country,roster_name,candidate_player_id,candidate_name,candidate_team
0,8,Algeria (ALG),GOUIRI Amine,4435,Amine Gouiri,OGC Nice | Rennes
1,12,Algeria (ALG),HADJAM Jaouen,45089,Jaouen Hadjam,Nantes
2,37,Argentina (ARG),RULLI Geronimo,6694,Gerónimo Rulli,Real Sociedad
3,50,Argentina (ARG),MEDINA Facundo,30415,Facundo Axel Medina,Lens
4,96,Austria (AUT),LJUBICIC Dejan,8623,Dejan Ljubicic,FC Köln


### Analyze Candidate Counts

In [64]:
candidate_counts = (
    flexible_candidates
    .groupby("roster_index")
    .size()
    .rename("candidate_count")
)

flexible_candidates = flexible_candidates.merge(
    candidate_counts,
    left_on="roster_index",
    right_index=True,
)

display(
    flexible_candidates[
        [
            "country",
            "roster_name",
            "candidate_name",
            "candidate_count",
        ]
    ]
    .sort_values(
        [
            "candidate_count",
            "country",
        ]
    )
    .head(50)
)

,country,roster_name,candidate_name,candidate_count
0,Algeria (ALG),GOUIRI Amine,Amine Gouiri,1
1,Algeria (ALG),HADJAM Jaouen,Jaouen Hadjam,1
2,Argentina (ARG),RULLI Geronimo,Gerónimo Rulli,1
3,Argentina (ARG),MEDINA Facundo,Facundo Axel Medina,1
4,Austria (AUT),LJUBICIC Dejan,Dejan Ljubicic,1
5,Austria (AUT),FRIEDL Marco,Marco Friedl,1
6,Bosnia And Herzegovina (BIH),KOLASINAC Sead,Sead Kolašinac,1
7,Bosnia And Herzegovina (BIH),DEMIROVIC Ermedin,Ermedin Demirović,1
13,Cabo Verde (CPV),NUNO DA,Nuno Miguel da Costa Jóia,1
14,Colombia (COL),PUERTA Gustavo,Gustavo Adolfo Puerta Molano,1


### Prepare Manual Review Dataset

In [65]:
review_candidates = (
    flexible_candidates
    .copy()
)

review_candidates["decision"] = ""

review_candidates["notes"] = ""

review_candidates.head()

,roster_index,country,roster_name,candidate_player_id,candidate_name,candidate_team,candidate_count,decision,notes
0,8,Algeria (ALG),GOUIRI Amine,4435,Amine Gouiri,OGC Nice | Rennes,1,,
1,12,Algeria (ALG),HADJAM Jaouen,45089,Jaouen Hadjam,Nantes,1,,
2,37,Argentina (ARG),RULLI Geronimo,6694,Gerónimo Rulli,Real Sociedad,1,,
3,50,Argentina (ARG),MEDINA Facundo,30415,Facundo Axel Medina,Lens,1,,
4,96,Austria (AUT),LJUBICIC Dejan,8623,Dejan Ljubicic,FC Köln,1,,


### Export Candidate Review

In [66]:
review_candidates.to_csv(
    PROCESSED_DIR /
    "flexible_name_candidates_v02.csv",
    index=False,
)

print(
    "Candidates exported:",
    len(review_candidates)
)

Candidates exported: 93


## Apply Decisions

### Load Reviewed Decisions

In [70]:
flexible_review = pd.read_csv(
    FLEXIBLE_REVIEW_PATH
)

print("Flexible review:", flexible_review.shape)

display(flexible_review.head())

Flexible review: (93, 10)


,roster_index,country,roster_name,candidate_player_id,candidate_name,candidate_team,candidate_count,review_decision,review_confidence,review_notes
0,8,Algeria (ALG),GOUIRI Amine,4435,Amine Gouiri,OGC Nice | Rennes,1,accept,high,Unique candidate with compatible full-name tok...
1,12,Algeria (ALG),HADJAM Jaouen,45089,Jaouen Hadjam,Nantes,1,accept,high,Unique candidate with compatible full-name tok...
2,37,Argentina (ARG),RULLI Geronimo,6694,Gerónimo Rulli,Real Sociedad,1,accept,high,Unique candidate with compatible full-name tok...
3,50,Argentina (ARG),MEDINA Facundo,30415,Facundo Axel Medina,Lens,1,accept,high,Unique candidate with compatible full-name tok...
4,96,Austria (AUT),LJUBICIC Dejan,8623,Dejan Ljubicic,FC Köln,1,accept,high,Unique candidate with compatible full-name tok...


### Checks

In [71]:
required_review_columns = {
    "roster_index",
    "candidate_player_id",
    "candidate_name",
    "candidate_team",
    "review_decision",
    "review_confidence",
    "review_notes",
}

missing_review_columns = (
    required_review_columns
    - set(flexible_review.columns)
)

assert not missing_review_columns, (
    "Missing columns in flexible review file: "
    f"{sorted(missing_review_columns)}"
)

print("Flexible review structural checks passed.")

Flexible review structural checks passed.


### Summary

In [72]:
flexible_review["review_decision"].value_counts(
    dropna=False
)

review_decision
accept          54
reject          35
needs_review     2
ambiguous        2
Name: count, dtype: int64

### Select Accepted Matches

In [73]:
accepted_flexible_matches = (
    flexible_review
    .loc[
        flexible_review["review_decision"] == "accept"
    ]
    .copy()
)

print(
    "Accepted candidate rows:",
    len(accepted_flexible_matches),
)

print(
    "Accepted roster players:",
    accepted_flexible_matches["roster_index"].nunique(),
)

Accepted candidate rows: 54
Accepted roster players: 54


### Validate Accepted Decisions

In [74]:
accepted_candidates_per_player = (
    accepted_flexible_matches
    .groupby("roster_index")
    .size()
)

multiple_accepted_candidates = (
    accepted_candidates_per_player[
        accepted_candidates_per_player > 1
    ]
)

assert multiple_accepted_candidates.empty, (
    "Some roster players have more than one accepted candidate:\n"
    f"{multiple_accepted_candidates}"
)

#### Validate existing indexes

In [75]:
unknown_roster_indices = (
    set(accepted_flexible_matches["roster_index"])
    - set(matched_rosters_v02.index)
)

assert not unknown_roster_indices, (
    "Some reviewed roster indices do not exist in "
    f"matched_rosters_v02: {unknown_roster_indices}"
)

#### Validate Still Unmatched

In [76]:
accepted_roster_indices = (
    accepted_flexible_matches["roster_index"]
    .astype(int)
)

already_matched_indices = (
    matched_rosters_v02.loc[
        accepted_roster_indices,
        "matched",
    ]
)

assert not already_matched_indices.any(), (
    "Some accepted flexible matches were already marked as matched."
)

#### Validate IDs in DB02

In [77]:
accepted_player_ids = (
    accepted_flexible_matches[
        "candidate_player_id"
    ]
    .astype(int)
)

missing_candidate_ids = (
    set(accepted_player_ids)
    - set(player_database_v02["player_id"].astype(int))
)

assert not missing_candidate_ids, (
    "Some accepted candidate player IDs are missing from "
    f"Player Database V02: {missing_candidate_ids}"
)

print("Accepted flexible match checks passed.")

Accepted flexible match checks passed.


### Apply Accepted Matches

In [78]:
accepted_lookup = (
    accepted_flexible_matches
    .set_index("roster_index")
)

In [79]:
accepted_indices = accepted_lookup.index.astype(int)

In [80]:
matched_rosters_v02.loc[
    accepted_indices,
    "player_id",
] = accepted_lookup[
    "candidate_player_id"
].to_numpy()

In [81]:
matched_rosters_v02.loc[
    accepted_indices,
    "database_name",
] = accepted_lookup[
    "candidate_name"
].to_numpy()

In [82]:
matched_rosters_v02.loc[
    accepted_indices,
    "database_team",
] = accepted_lookup[
    "candidate_team"
].to_numpy()

In [83]:
matched_rosters_v02.loc[
    accepted_indices,
    "matched",
] = True

In [84]:
matched_rosters_v02.loc[
    accepted_indices,
    "match_method",
] = "v02_validated_flexible_match"

In [85]:
matched_rosters_v02.loc[
    accepted_indices,
    "review_confidence",
] = accepted_lookup[
    "review_confidence"
].to_numpy()

In [86]:
matched_rosters_v02.loc[
    accepted_indices,
    "review_notes",
] = accepted_lookup[
    "review_notes"
].to_numpy()

### Post-Update Integrity Checks

In [87]:
assert matched_rosters_v02.loc[
    accepted_indices,
    "matched",
].all()

In [88]:
assert matched_rosters_v02.loc[
    accepted_indices,
    "player_id",
].notna().all()

In [89]:
assert matched_rosters_v02.loc[
    accepted_indices,
    "database_name",
].notna().all()

In [90]:
assert matched_rosters_v02.loc[
    accepted_indices,
    "database_name",
].notna().all()

In [91]:
assert len(matched_rosters_v02) == len(matching_v01), (
    "Applying reviewed matches changed the roster size."
)

## Coverage Summary

In [92]:
matched_v01_count = (
    matching_v01["matched"].sum()
)

matched_v02_count = (
    matched_rosters_v02["matched"].sum()
)

newly_recovered_count = (
    matched_v02_count
    - matched_v01_count
)

unmatched_v02_count = (
    (~matched_rosters_v02["matched"]).sum()
)

coverage_v01 = (
    matching_v01["matched"].mean()
)

coverage_v02 = (
    matched_rosters_v02["matched"].mean()
)

In [93]:
print("Validated flexible matches applied.")
print(f"V01 matched players: {matched_v01_count}")
print(f"Newly recovered players: {newly_recovered_count}")
print(f"V02 matched players: {matched_v02_count}")
print(f"V02 unmatched players: {unmatched_v02_count}")
print(f"V01 coverage: {coverage_v01:.2%}")
print(f"V02 coverage: {coverage_v02:.2%}")
print(
    "Coverage improvement: "
    f"{coverage_v02 - coverage_v01:.2%}"
)

Validated flexible matches applied.
V01 matched players: 577
Newly recovered players: 54
V02 matched players: 631
V02 unmatched players: 617
V01 coverage: 46.23%
V02 coverage: 50.56%
Coverage improvement: 4.33%


In [94]:
coverage_summary_v01_v02 = pd.DataFrame(
    {
        "metric": [
            "Matched players",
            "Unmatched players",
            "Coverage",
        ],
        "v01": [
            matched_v01_count,
            (~matching_v01["matched"]).sum(),
            coverage_v01,
        ],
        "v02": [
            matched_v02_count,
            unmatched_v02_count,
            coverage_v02,
        ],
    }
)

coverage_summary_v01_v02["improvement"] = (
    coverage_summary_v01_v02["v02"]
    - coverage_summary_v01_v02["v01"]
)

coverage_summary_v01_v02

,metric,v01,v02,improvement
0,Matched players,577.00000,631.000000,54.000000
1,Unmatched players,671.00000,617.000000,-54.000000
2,Coverage,0.46234,0.505609,0.043269


## Country-Level Coverage Comparison

### Country Coverage

In [95]:
country_coverage_v01 = (
    matching_v01
    .groupby(
        ["country", "country_clean"],
        as_index=False,
    )
    .agg(
        roster_size=("roster_name", "size"),
        matched_v01=("matched", "sum"),
    )
)

country_coverage_v01["unmatched_v01"] = (
    country_coverage_v01["roster_size"]
    - country_coverage_v01["matched_v01"]
)

country_coverage_v01["coverage_v01"] = (
    country_coverage_v01["matched_v01"]
    / country_coverage_v01["roster_size"]
)

In [96]:
country_coverage_v02 = (
    matched_rosters_v02
    .groupby(
        ["country", "country_clean"],
        as_index=False,
    )
    .agg(
        roster_size=("roster_name", "size"),
        matched_v02=("matched", "sum"),
    )
)

country_coverage_v02["unmatched_v02"] = (
    country_coverage_v02["roster_size"]
    - country_coverage_v02["matched_v02"]
)

country_coverage_v02["coverage_v02"] = (
    country_coverage_v02["matched_v02"]
    / country_coverage_v02["roster_size"]
)

### Merge Versions

In [97]:
country_coverage_comparison = (
    country_coverage_v01
    .merge(
        country_coverage_v02,
        on=["country", "country_clean", "roster_size"],
        how="outer",
        validate="one_to_one",
    )
)

#### Add Upgrades

In [98]:
country_coverage_comparison["matched_gain"] = (
    country_coverage_comparison["matched_v02"]
    - country_coverage_comparison["matched_v01"]
)

country_coverage_comparison["coverage_gain"] = (
    country_coverage_comparison["coverage_v02"]
    - country_coverage_comparison["coverage_v01"]
)

#### Sort

In [99]:
country_coverage_comparison = (
    country_coverage_comparison
    .sort_values(
        [
            "matched_gain",
            "coverage_gain",
            "country_clean",
        ],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

#### Display

In [100]:
display(
    country_coverage_comparison[
        [
            "country_clean",
            "roster_size",
            "matched_v01",
            "matched_v02",
            "matched_gain",
            "coverage_v01",
            "coverage_v02",
            "coverage_gain",
        ]
    ]
)

,country_clean,roster_size,matched_v01,matched_v02,matched_gain,coverage_v01,coverage_v02,coverage_gain
0,France,26,15,22,7,0.576923,0.846154,0.269231
1,Germany,26,15,22,7,0.576923,0.846154,0.269231
2,Côte D'Ivoire,26,13,17,4,0.500000,0.653846,0.153846
3,Haiti,26,0,3,3,0.000000,0.115385,0.115385
4,Tunisia,26,8,11,3,0.307692,0.423077,0.115385
5,Senegal,26,16,19,3,0.615385,0.730769,0.115385
6,Argentina,26,18,20,2,0.692308,0.769231,0.076923
7,Austria,26,18,20,2,0.692308,0.769231,0.076923
8,Colombia,26,18,20,2,0.692308,0.769231,0.076923
9,Spain,26,19,21,2,0.730769,0.807692,0.076923


In [101]:
country_coverage_display = (
    country_coverage_comparison[
        [
            "country_clean",
            "roster_size",
            "matched_v01",
            "matched_v02",
            "matched_gain",
            "coverage_v01",
            "coverage_v02",
            "coverage_gain",
        ]
    ]
    .copy()
)

for column in [
    "coverage_v01",
    "coverage_v02",
    "coverage_gain",
]:
    country_coverage_display[column] = (
        country_coverage_display[column]
        .map(lambda value: f"{value:.2%}")
    )

country_coverage_display

,country_clean,roster_size,matched_v01,matched_v02,matched_gain,coverage_v01,coverage_v02,coverage_gain
0,France,26,15,22,7,57.69%,84.62%,26.92%
1,Germany,26,15,22,7,57.69%,84.62%,26.92%
2,Côte D'Ivoire,26,13,17,4,50.00%,65.38%,15.38%
3,Haiti,26,0,3,3,0.00%,11.54%,11.54%
4,Tunisia,26,8,11,3,30.77%,42.31%,11.54%
5,Senegal,26,16,19,3,61.54%,73.08%,11.54%
6,Argentina,26,18,20,2,69.23%,76.92%,7.69%
7,Austria,26,18,20,2,69.23%,76.92%,7.69%
8,Colombia,26,18,20,2,69.23%,76.92%,7.69%
9,Spain,26,19,21,2,73.08%,80.77%,7.69%


#### Checks

In [102]:
assert len(country_coverage_comparison) == 48, (
    "Expected coverage results for 48 national teams."
)

assert (
    country_coverage_comparison["roster_size"] == 26
).all(), (
    "Each national team should contain 26 roster players."
)

assert (
    country_coverage_comparison["matched_gain"] >= 0
).all(), (
    "V02 should not lose previously validated matches."
)

assert (
    country_coverage_comparison["matched_v02"].sum()
    == matched_rosters_v02["matched"].sum()
)

print("Country-level coverage checks passed.")

Country-level coverage checks passed.


## Analyze Remaining Unmatched Players

In [103]:
final_unmatched_v02 = (
    matched_rosters_v02
    .loc[~matched_rosters_v02["matched"]]
    .copy()
)

print("Final unmatched players:", len(final_unmatched_v02))

Final unmatched players: 617


In [104]:
remaining_unmatched_by_country = (
    final_unmatched_v02
    .groupby(
        ["country", "country_clean"],
        as_index=False,
    )
    .agg(
        unmatched_players=("roster_name", "size")
    )
    .sort_values(
        ["unmatched_players", "country_clean"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

remaining_unmatched_by_country

,country,country_clean,unmatched_players
0,Curaçao (CUW),Curaçao,26
1,Iraq (IRQ),Iraq,26
2,Jordan (JOR),Jordan,26
3,New Zealand (NZL),New Zealand,26
4,Uzbekistan (UZB),Uzbekistan,26
5,Bosnia And Herzegovina (BIH),Bosnia And Herzegovina,24
6,Norway (NOR),Norway,24
7,Haiti (HAI),Haiti,23
8,Sweden (SWE),Sweden,22
9,Australia (AUS),Australia,18


# Export Final Outputs

## Current Roster Matching

#### Duplicate Player Resolution (Notebook 17)

A duplicated historical player identifier was detected in the Brazilian roster.

Both `DANILO Danilo` and `Danilo Luiz da Silva` were assigned to player ID `3063`, representing the same historical player. The exact normalized match was retained, while the less precise token-subset match was removed to prevent duplicate aggregation in downstream team vectors.

In [126]:
duplicate_resolution_mask = (
    (matched_rosters_v02["country_clean"] == "Brazil")
    & (
        matched_rosters_v02["roster_name"]
        .astype(str)
        .str.strip()
        .eq("DANILO Danilo")
    )
)

assert duplicate_resolution_mask.sum() == 1, (
    "The expected duplicated Brazilian roster entry was not found "
    "exactly once."
)

matched_rosters_v02.loc[
    duplicate_resolution_mask,
    ["player_id", "database_name", "database_team"]
] = pd.NA

matched_rosters_v02.loc[
    duplicate_resolution_mask,
    "matched"
] = False

matched_rosters_v02.loc[
    duplicate_resolution_mask,
    "match_method"
] = "duplicate_roster_entry"

matched_rosters_v02.loc[
    duplicate_resolution_mask,
    "review_notes"
] = (
    "Duplicate roster entry assigned to player_id 3063. "
    "The exact normalized match was retained."
)

In [127]:
assert (
    matched_rosters_v02
    .loc[matched_rosters_v02["matched"], "player_id"]
    .is_unique
), "Matched player IDs must be unique."

assert len(matched_rosters_v02) == 1248, (
    "The roster matching dataset must preserve all 1248 roster entries."
)

assert (
    matched_rosters_v02
    .groupby("country_clean")
    .size()
    .eq(26)
    .all()
), "Every country must preserve exactly 26 roster entries."

print("Duplicate player resolved successfully.")
print("Matched player IDs are unique.")

Duplicate player resolved successfully.
Matched player IDs are unique.


In [128]:
matched_rosters_v02.to_csv(
    OUTPUT_MATCHING_PATH,
    index=False,
)

### Notebook 16 Export

In [105]:
matched_rosters_v02["player_id"] = (
    matched_rosters_v02["player_id"]
    .astype("Int64")
)

In [106]:
matched_rosters_v02 = (
    matched_rosters_v02
    .sort_values(
        ["country_clean", "shirt_number", "roster_name"]
    )
    .reset_index(drop=True)
)

In [107]:
matched_rosters_v02.to_csv(
    OUTPUT_MATCHING_PATH,
    index=False,
)

print(f"Matching V02 exported to: {OUTPUT_MATCHING_PATH}")
print("Shape:", matched_rosters_v02.shape)

Matching V02 exported to: ..\data\processed\current_roster_matching_v02.csv
Shape: (1248, 12)


## Final Unmatched Players

In [108]:
final_unmatched_v02 = (
    matched_rosters_v02
    .loc[~matched_rosters_v02["matched"]]
    .copy()
)

In [109]:
final_unmatched_v02.to_csv(
    OUTPUT_UNMATCHED_PATH,
    index=False,
)

print(f"Final unmatched players exported to: {OUTPUT_UNMATCHED_PATH}")
print("Shape:", final_unmatched_v02.shape)

Final unmatched players exported to: ..\data\processed\unmatched_roster_players_v02.csv
Shape: (617, 12)


## Overall Coverage Report

In [110]:
coverage_report_v02 = pd.DataFrame(
    {
        "version": ["v01", "v02"],
        "roster_players": [
            len(matching_v01),
            len(matched_rosters_v02),
        ],
        "matched_players": [
            matching_v01["matched"].sum(),
            matched_rosters_v02["matched"].sum(),
        ],
        "unmatched_players": [
            (~matching_v01["matched"]).sum(),
            (~matched_rosters_v02["matched"]).sum(),
        ],
        "coverage": [
            matching_v01["matched"].mean(),
            matched_rosters_v02["matched"].mean(),
        ],
    }
)

coverage_report_v02

,version,roster_players,matched_players,unmatched_players,coverage
0,v01,1248,577,671,0.462340
1,v02,1248,631,617,0.505609


In [111]:
coverage_report_v02.to_csv(
    OUTPUT_COVERAGE_PATH,
    index=False,
)

print(f"Coverage report exported to: {OUTPUT_COVERAGE_PATH}")

Coverage report exported to: ..\data\processed\roster_coverage_report_v02.csv


## Country Comparison

In [112]:
country_coverage_comparison.to_csv(
    OUTPUT_COMPARISON_PATH,
    index=False,
)

print(
    "Country coverage comparison exported to:",
    OUTPUT_COMPARISON_PATH,
)

Country coverage comparison exported to: ..\data\processed\current_roster_matching_comparison_v01_v02.csv


## Final Export Checks

In [119]:
assert OUTPUT_MATCHING_PATH.exists()
assert OUTPUT_UNMATCHED_PATH.exists()
assert OUTPUT_COVERAGE_PATH.exists()
assert OUTPUT_COMPARISON_PATH.exists()

assert len(matched_rosters_v02) == 1248
assert matched_rosters_v02["matched"].sum() == 631
assert len(final_unmatched_v02) == 617

assert matched_rosters_v02.loc[
    matched_rosters_v02["matched"],
    "player_id",
].notna().all()

assert matched_rosters_v02.loc[
    ~matched_rosters_v02["matched"],
    "player_id",
].isna().all()

print("All final export checks passed.")

All final export checks passed.


# Discussion

### English

- The expansion of the International Player Database produced a measurable improvement in current roster coverage. Overall player identification increased from 46.23% to 50.56%, recovering 54 additional players through a structured manual validation workflow.

- Coverage gains were concentrated in national teams whose players participate extensively in club competitions already included in the database expansion. France and Germany showed the largest improvements, while several African and South American teams also benefited from the additional historical information.

- Conversely, multiple national teams exhibited little or no improvement. Countries such as Curaçao, Iraq, Jordan, New Zealand, and Uzbekistan remain entirely unmatched, suggesting that the primary limitation is no longer the matching strategy itself but the absence of historical player data within the current database.

- The introduction of a candidate-generation stage followed by explicit human validation proved to be an effective compromise between automation and reliability. Rather than increasing the complexity of automatic matching rules, the notebook now produces reproducible candidate lists that can be reviewed before incorporation into the final dataset, preserving both accuracy and auditability.

### Español

- La expansión de la International Player Database produjo una mejora cuantificable en la cobertura de los planteles actuales. La identificación de jugadores aumentó del 46.23% al 50.56%, recuperando 54 jugadores adicionales mediante un flujo de validación manual estructurado.

- Las mayores mejoras se concentraron en selecciones cuyos futbolistas participan ampliamente en competiciones de clubes ya incorporadas a la base de datos. Francia y Alemania registraron los incrementos más importantes, mientras que varias selecciones africanas y sudamericanas también se beneficiaron de la información histórica adicional.

- Por el contrario, múltiples selecciones presentaron poca o ninguna mejora. Países como Curaçao, Iraq, Jordania, Nueva Zelanda y Uzbekistán continúan completamente sin representación, lo que indica que la principal limitación ya no es la estrategia de matching sino la ausencia de información histórica dentro de la base de datos actual.

- La incorporación de una etapa de generación de candidatos seguida de una validación humana explícita resultó ser un compromiso efectivo entre automatización y confiabilidad. En lugar de incrementar la complejidad de las reglas automáticas de matching, la notebook genera listas reproducibles de candidatos que pueden revisarse antes de incorporarse al dataset final, preservando tanto la precisión como la trazabilidad del proceso.

# Conclusion

### English

- Current Roster Matching V02 successfully integrated the expanded International Player Database into the player identification pipeline.
 
- A reproducible review workflow was introduced by combining automatic candidate generation with explicit manual validation, allowing 54 additional players to be identified without compromising matching quality.
 
- Overall roster coverage increased from 46.23% to 50.56%, with the largest improvements observed in countries whose players were represented in the newly incorporated club competitions.

- The remaining unmatched players now provide a clear roadmap for future database expansion, indicating which national teams require additional historical competitions or alternative data sources rather than - further modifications to the matching algorithm.

- This notebook establishes the finalized current roster matching dataset that will serve as the input for constructing Current Team Vectors in the next stage of the project.

### Español

- Current Roster Matching V02 integró exitosamente la International Player Database expandida dentro del pipeline de identificación de jugadores.

- Se incorporó un flujo de revisión reproducible que combina generación automática de candidatos con validación manual explícita, permitiendo identificar 54 jugadores adicionales sin comprometer la calidad del matching.

- La cobertura global de los planteles aumentó del 46.23% al 50.56%, observándose las mayores mejoras en aquellas selecciones cuyos jugadores estaban representados en las nuevas competiciones de clubes incorporadas.

- Los jugadores que permanecen sin identificar constituyen ahora una hoja de ruta clara para futuras expansiones de la base histórica, indicando qué selecciones requieren nuevas competiciones o fuentes alternativas de datos antes que modificaciones adicionales al algoritmo de matching.

- Esta notebook establece el dataset definitivo de Current Roster Matching que será utilizado como entrada para la construcción de los Current Team Vectors en la siguiente etapa del proyecto.